# 03 — Fairness Metrics

**FairLens AI / NyayaLens — Person 3 (ML Layer)**

This notebook measures how the baseline model's performance differs across
gender groups using **Fairlearn MetricFrame**.

### What we do here
1. Rebuild the preprocessed data and trained model from Notebook 02
2. Use Fairlearn MetricFrame to compute accuracy and recall per group
3. Compute the max difference across groups
4. Classify severity: HIGH (>10%), MEDIUM (5-10%), LOW (<5%)
5. Visualize per-group metrics

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_openml
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Fairlearn MetricFrame: computes any sklearn metric separately for each
# unique value of a sensitive feature (e.g., Male vs Female).
from fairlearn.metrics import MetricFrame

sns.set_theme(style="whitegrid")
print("Libraries loaded.")

## Step 1 — Rebuild data and model

We repeat the preprocessing and training from Notebook 02 so this notebook
is self-contained. In production, the `ml/src/` modules handle this.

In [ ]:
# --- Load & clean ---
raw_df = fetch_openml("adult", version=2, as_frame=True).frame
cleaned_df = raw_df.copy()
for col in cleaned_df.columns:
    if cleaned_df[col].dtype == object:
        cleaned_df = cleaned_df[cleaned_df[col] != "?"]
cleaned_df = cleaned_df.dropna()

income_labels = cleaned_df["class"].apply(
    lambda v: 1 if ">50K" in str(v) else 0
)
gender_sensitive = cleaned_df["sex"].copy()

feature_df = cleaned_df.drop(columns=["sex", "race", "fnlwgt", "class"])
feature_df = pd.get_dummies(feature_df, drop_first=True)

features_train, features_test, labels_train, labels_test = train_test_split(
    feature_df, income_labels, test_size=0.2, random_state=42
)

scaler = StandardScaler()
features_train = pd.DataFrame(
    scaler.fit_transform(features_train),
    columns=features_train.columns, index=features_train.index
)
features_test = pd.DataFrame(
    scaler.transform(features_test),
    columns=features_test.columns, index=features_test.index
)

gender_train = gender_sensitive.loc[features_train.index]
gender_test = gender_sensitive.loc[features_test.index]

# --- Train baseline ---
baseline_model = LogisticRegression(max_iter=5000, random_state=42)
baseline_model.fit(features_train, labels_train)
baseline_predictions = baseline_model.predict(features_test)

print(f"Baseline accuracy: {accuracy_score(labels_test, baseline_predictions):.4f}")
print("Data and model ready.")

## Step 2 — Fairlearn MetricFrame

**What is MetricFrame?**

You give it:
- A dict of metric functions (e.g., `accuracy_score`, `recall_score`)
- The true labels (`y_true`)
- The predicted labels (`y_pred`)
- A sensitive feature column (`sensitive_features`)

It computes each metric separately for each unique group in the sensitive feature.

For example, if `sensitive_features` is the `sex` column with values `Male` and `Female`,
MetricFrame will compute accuracy for Males and accuracy for Females separately.

In [ ]:
# Create a MetricFrame that computes accuracy and recall per gender group
fairness_frame = MetricFrame(
    metrics={
        "accuracy": accuracy_score,
        "recall":   recall_score,
    },
    y_true=labels_test,
    y_pred=baseline_predictions,
    sensitive_features=gender_test,
)

print("=== Per-group metrics ===")
print(fairness_frame.by_group)
print()
print("=== Overall metrics ===")
print(fairness_frame.overall)

## Step 3 — Metric differences (disparity)

`.difference()` computes the **maximum gap** between any two groups for each metric.
This is the primary measure of bias.

In [ ]:
differences = fairness_frame.difference()
print("Metric differences (max gap between groups):")
print(f"  Accuracy difference:  {differences['accuracy']:.4f}")
print(f"  Recall difference:    {differences['recall']:.4f}")

## Step 4 — Severity classification

Our severity rules:
- **HIGH**: difference > 10%
- **MEDIUM**: difference between 5% and 10%
- **LOW**: difference < 5%

In [ ]:
for metric_name, diff_value in differences.items():
    if diff_value > 0.10:
        severity = "HIGH"
    elif diff_value >= 0.05:
        severity = "MEDIUM"
    else:
        severity = "LOW"
    
    print(f"{metric_name} difference: {diff_value:.4f} -> Severity: {severity}")

## Step 5 — Visualize per-group metrics

A grouped bar chart makes it easy to see which group the model favours.

In [ ]:
per_group = fairness_frame.by_group

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Accuracy per group
per_group["accuracy"].plot(kind="bar", ax=axes[0], color=["#FF9800", "#2196F3"])
axes[0].set_title("Accuracy by Gender")
axes[0].set_ylabel("Accuracy")
axes[0].set_ylim(0.5, 1.0)
axes[0].axhline(y=fairness_frame.overall["accuracy"], color="gray",
                linestyle="--", label="Overall")
axes[0].legend()
plt.sca(axes[0])
plt.xticks(rotation=0)

# Recall per group
per_group["recall"].plot(kind="bar", ax=axes[1], color=["#FF9800", "#2196F3"])
axes[1].set_title("Recall by Gender")
axes[1].set_ylabel("Recall")
axes[1].set_ylim(0.0, 1.0)
axes[1].axhline(y=fairness_frame.overall["recall"], color="gray",
                linestyle="--", label="Overall")
axes[1].legend()
plt.sca(axes[1])
plt.xticks(rotation=0)

plt.suptitle("Baseline Model: Per-Group Fairness Metrics", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Summary

- **Accuracy difference** across gender: ~11% (HIGH severity).
  - Female accuracy is much higher (fewer false positives) but this is
    partly because the model predicts <=50K for most Females.
- **Recall difference** across gender: ~9% (MEDIUM severity).
  - The model catches more Male >50K earners than Female >50K earners.
  - This is the most actionable bias signal: qualified Female applicants
    are being missed at a higher rate.
- This confirms that fairness mitigation is needed.

**Next:** Notebook 04 — Bias Mitigation